# 🛒 E-Commerce Sales — Exploratory Data Analysis

> **Dataset:** Synthetic e-commerce orders dataset (5,000 rows, 2023)  
> **Goal:** Understand sales trends, top categories, customer behaviour, and identify business insights.

## Table of Contents
1. [Import Libraries & Load Data](#1)
2. [Data Overview & Quality Check](#2)
3. [Monthly Revenue Trend](#3)
4. [Category & Product Performance](#4)
5. [City-wise Analysis](#5)
6. [Payment Methods](#6)
7. [Customer Ratings](#7)
8. [Return Rate Analysis](#8)
9. [Discount vs Revenue Heatmap](#9)
10. [Quarterly Revenue](#10)
11. [Key Business Insights](#11)

In [ ]:
# ── 1. Import Libraries & Load Data ──────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.titleweight': 'bold'})
COLORS = sns.color_palette('muted', 10)

df = pd.read_csv('data/ecommerce_sales.csv', parse_dates=['order_date'])
df['month_num']  = df['order_date'].dt.month
df['quarter']    = df['order_date'].dt.to_period('Q').astype(str)

print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── 2. Data Overview & Quality Check ─────────────────────────────
print('=== DATA TYPES ===')
print(df.dtypes)
print('\n=== MISSING VALUES ===')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print('\n=== BASIC STATS ===')
df[['unit_price','quantity','discount','revenue','rating']].describe().round(2)

In [ ]:
# ── 3. Monthly Revenue Trend ──────────────────────────────────────
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month_num').agg(revenue=('revenue','sum'), orders=('order_id','count')).reset_index()
monthly['month_label'] = monthly['month_num'].apply(lambda x: month_labels[x-1])

fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()
ax1.bar(monthly['month_label'], monthly['revenue']/1000, color=COLORS[0], alpha=0.75, label='Revenue (₹K)')
ax2.plot(monthly['month_label'], monthly['orders'], color=COLORS[3], marker='o', linewidth=2, label='Orders')
ax1.set_ylabel('Revenue (₹ Thousands)')
ax2.set_ylabel('Number of Orders')
ax1.set_title('Monthly Revenue & Order Volume — 2023')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.show()

print(f"Best month : {monthly.loc[monthly['revenue'].idxmax(), 'month_label']}")
print(f"Worst month: {monthly.loc[monthly['revenue'].idxmin(), 'month_label']}")

In [ ]:
# ── 4. Category & Product Performance ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_rev = df.groupby('category')['revenue'].sum().sort_values()
axes[0].barh(cat_rev.index, cat_rev.values/1000, color=COLORS[:len(cat_rev)])
axes[0].set_xlabel('Revenue (₹K)')
axes[0].set_title('Revenue by Category')

top_p = df.groupby('product')['revenue'].sum().nlargest(10).sort_values()
axes[1].barh(top_p.index, top_p.values/1000, color=COLORS[1])
axes[1].set_xlabel('Revenue (₹K)')
axes[1].set_title('Top 10 Products')

plt.tight_layout()
plt.show()

In [ ]:
# ── 5. City-wise Analysis ─────────────────────────────────────────
city = df.groupby('city').agg(orders=('order_id','count'), revenue=('revenue','sum')).sort_values('revenue', ascending=False).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(city['city'], city['orders'], color=COLORS[2])
axes[0].set_title('Orders by City'); axes[0].tick_params(axis='x', rotation=30)
axes[1].bar(city['city'], city['revenue']/1000, color=COLORS[4])
axes[1].set_title('Revenue by City (₹K)'); axes[1].tick_params(axis='x', rotation=30)
plt.suptitle('City-wise Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Top city: {city.iloc[0]['city']}")

In [ ]:
# ── 6. Payment Methods ────────────────────────────────────────────
payment = df['payment_method'].value_counts()
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(payment.values, labels=payment.index, autopct='%1.1f%%', colors=COLORS[:len(payment)], startangle=140)
ax.set_title('Payment Method Distribution')
plt.tight_layout()
plt.show()
print(f"Most popular: {payment.idxmax()} — {payment.max()/len(df)*100:.1f}%")

In [ ]:
# ── 7. Customer Ratings ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(df['rating'], bins=25, color=COLORS[5], edgecolor='white')
axes[0].axvline(df['rating'].mean(), color='red', linestyle='--', label=f'Mean: {df["rating"].mean():.2f}')
axes[0].set_title('Rating Distribution'); axes[0].legend()

cat_rating = df.groupby('category')['rating'].mean().sort_values(ascending=False)
axes[1].bar(cat_rating.index, cat_rating.values, color=COLORS[6])
axes[1].set_ylim(0, 5.5); axes[1].set_title('Avg Rating by Category')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8. Return Rate by Category ────────────────────────────────────
return_rate = df.groupby('category')['returned'].mean().sort_values(ascending=False) * 100
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(return_rate.index, return_rate.values, color=COLORS[3])
ax.axhline(return_rate.mean(), color='red', linestyle='--', label=f'Avg: {return_rate.mean():.1f}%')
ax.set_ylabel('Return Rate (%)'); ax.set_title('Return Rate by Category')
ax.tick_params(axis='x', rotation=20); ax.legend()
plt.tight_layout()
plt.show()
print(f"Highest: {return_rate.idxmax()} at {return_rate.max():.1f}%")

In [ ]:
# ── 9. Discount vs Revenue Heatmap ───────────────────────────────
df['discount_pct'] = (df['discount'] * 100).astype(int).astype(str) + '%'
pivot = df.groupby(['category','discount_pct'])['revenue'].sum().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(pivot/1000, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Revenue (₹K)'})
ax.set_title('Revenue Heatmap — Category vs Discount Level')
plt.tight_layout()
plt.show()

In [ ]:
# ── 10. Quarterly Revenue ─────────────────────────────────────────
qtr = df.groupby('quarter')['revenue'].sum().reset_index()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(qtr['quarter'], qtr['revenue']/1000, color=COLORS[:4])
ax.set_ylabel('Revenue (₹K)'); ax.set_title('Quarterly Revenue — 2023')
for i, row in qtr.iterrows():
    ax.text(i, row['revenue']/1000+10, f'₹{row["revenue"]/1000:.0f}K', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Key Business Insights 🔍

| # | Insight | Detail |
|---|---------|--------|
| 1 | **Electronics dominates revenue** | Electronics accounts for the highest revenue share (~₹20L) despite being one of the smaller volume categories — driven by high unit prices. |
| 2 | **December is peak month** | Revenue spikes in December — likely due to holiday/festival season shopping. Inventory planning should account for this. |
| 3 | **UPI & Credit Card are preferred** | Together they account for ~60% of payments. UPI adoption is strong, reflecting Indian market trends. |
| 4 | **Sports has highest return rate** | Sports products see the most returns (~12.7%). Could indicate sizing issues or quality concerns. |
| 5 | **Ratings are fairly consistent** | Average rating of ~3.74 across all categories. No single category has a dramatically lower score. |
| 6 | **Hyderabad leads in revenue** | Among all cities, Hyderabad generates the highest revenue. Could be a target for regional marketing campaigns. |
| 7 | **10% discount drives most orders** | The heatmap shows the ₹ value peaks at 10% discount across most categories — sweet spot for promotions. |